# Liver dataset selection — audit (`_repaired`)

**v3 — poprawka błędu z v2.** Tak jak w `kidney_dataset_repaired.ipynb`: v2 użył `condition="Wildtype"` zamiast `condition=["Wildtype", "Wtype", "N/A"]", którego użyłem dla brain — to pomijało wszystkie datasety z `condition="N/A"` (82 zamiast 112 kandydatów). Ten notebook używa teraz dokładnie tego samego filtra biologicznego co brain, konsekwentnie w zapytaniu szerokim i finalnym. `mz_min=200, mz_max=1400` bez zmian — osobna oś, nierozwiązywana tu.

Reszta metodyki bez zmian: `DatasetExplorer.review_current(profile="liver")`/`.apply_review()`, zero zmian w bibliotece, zero nowych pobrań surowych danych.

In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root

PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
import json

import pandas as pd
from IPython.display import display

from msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is 12.08.2026 (DD, MM, YYYY) -- same cache as the other two notebooks.
explorer = DatasetExplorer(
    source="metaspace",
    cache_dir="assets/local/datasets/metaspace",
    refresh_cache=False,
)

## 1. Szeroka pula kandydatów — dokładnie filtr biologiczny z `brain_dataset.ipynb`

`condition=["Wildtype", "Wtype", "N/A"]`. Podnosi pulę z 82 (v2, błędne) do 112 kandydatów.

In [3]:
broad_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
}
results = explorer.filter(broad_filters)
print(f"Found {len(results)} datasets")
display(results[["dataset_id", "name", "condition", "analyzer_type", "mz_min", "mz_max", "pixel_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

Found 112 datasets


,dataset_id,name,condition,analyzer_type,mz_min,mz_max,pixel_count
0,2026-06-22_15h21m26s,WT_D12_3-neg,Wildtype,TOF,7.500978e+01,1199.833220,13964
1,2026-06-22_15h19m59s,WT_D12_2-neg,Wildtype,TOF,7.500978e+01,1199.833220,5142
2,2026-06-22_15h19m49s,WT_D12_1-neg,Wildtype,TOF,7.500978e+01,1199.833220,9002
3,2026-06-22_15h19m03s,WT_D5_3-neg,Wildtype,TOF,7.500978e+01,1199.833220,9488
4,2026-06-22_15h17m46s,WT_D5_2-neg,Wildtype,TOF,7.500978e+01,1199.833220,7172
...,...,...,...,...,...,...,...
107,2017-09-08_10h22m41s,testliver1,N/A,FTICR,3.024644e+02,1649.959961,10489
108,2017-10-04_12h53m56s,testliver2_rms,N/A,FTICR,5.700283e+01,987.500854,11458
109,2018-02-12_14h15m39s,201802_leber_cg-rms,Wildtype,FTICR,3.628341e-309,1099.451660,22789
110,2018-02-21_12h25m02s,20180221__ROI02_Liver_PA,N/A,FTICR,6.368766e+01,1099.726685,687


## 2. Ładowanie dotychczasowej selekcji i przeniesienie jej wykluczeń do sesji

`liver/filter.json` ma `exclude_dataset_ids: []`. Wszystkie 18 obecnie zaakceptowanych ID muszą się mieścić w nowej puli 112 (sprawdzone niżej).

In [4]:
existing_filter = json.load(open("data/liver_workspace/configs/datasets/liver/filter.json"))
existing_selection = json.load(open("data/liver_workspace/configs/datasets/liver/selection.json"))
existing_excluded_ids = existing_filter.get("exclude_dataset_ids", [])
existing_selected_ids = set(existing_selection["dataset_ids"])
print(f"existing selection: {len(existing_selected_ids)} selected, {len(existing_excluded_ids)} manually excluded")

known_ids = set(results["dataset_id"].astype(str))
missing = (set(existing_excluded_ids) | existing_selected_ids) - known_ids
assert not missing, f"existing IDs missing from the new broad pool: {missing}"

explorer.exclude(existing_excluded_ids)

existing selection: 18 selected, 0 manually excluded


,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-06-22_15h21m26s,WT_D12_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-06-22_15h19m59s,WT_D12_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-06-22_15h19m49s,WT_D12_1-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-06-22_15h19m03s,WT_D5_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-06-22_15h17m46s,WT_D5_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,2017-09-08_10h22m41s,testliver1,metaspace,None,https://metaspace2020.eu/dataset/2017-09-08_10...,Mus musculus (mouse),Liver,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
108,2017-10-04_12h53m56s,testliver2_rms,metaspace,None,https://metaspace2020.eu/dataset/2017-10-04_12...,Mus musculus (mouse),Liver,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False
109,2018-02-12_14h15m39s,201802_leber_cg-rms,metaspace,None,https://metaspace2020.eu/dataset/2018-02-12_14...,Mus musculus (mouse),Liver,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
110,2018-02-21_12h25m02s,20180221__ROI02_Liver_PA,metaspace,None,https://metaspace2020.eu/dataset/2018-02-21_12...,Mus musculus (mouse),Liver,N/A,N/A,,...,None,None,0.1,None,None,None,None,None,,False


## 3. Ręczna kontrola jakości, której żadna reguła biblioteczna nie łapie

Ten sam skan co w kidney, dla kompletności (species/tissue-name mismatch, jawne oznaczenia testowe):

In [5]:
suspect_pattern = r"zebrafish|drosophila|\brat\b|\bhuman\b|\btest\b|\(test\)|calib|standard"
suspects = results[results["name"].str.contains(suspect_pattern, case=False, na=False, regex=True)]
display(suspects[["dataset_id", "name", "condition", "organisms", "pixel_count"]])

,dataset_id,name,condition,organisms,pixel_count
104,2018-07-11_14h27m30s,JuS_mouse liver_100x100 10um DAN-test,N/A,Mus musculus (mouse),10000


`JuS_mouse liver_100x100 10um DAN-test` zawiera `test`, ale nazwa mówi wprost "mouse liver" — brak niedopasowania gatunku/tkanki, "DAN-test" czyta się jako test matrycy DAN na prawdziwej tkance, nie jako skan kalibracyjny bez tkanki. **Nie wykluczam ręcznie** — flaguję tylko jako sprawdzone. (I tak nie wchodzi do finalnej listy — nie mieści się w zakresie `200–1400`, patrz sekcja 5.)

## 4. Przegląd biblioteczny (`DatasetExplorer.review_current`, `profile="liver"`)

In [6]:
review = explorer.review_current(profile="liver")

print("available rules:", review.available_rules)
display(review.summary())

display(
    review.table.loc[
        review.table["duplicate_cluster_size"] > 1,
        ["duplicate_cluster_id", "dataset_id", "name", "pixel_count", "duplicate_confidence", "duplicate_excluded", "recommended_keeper_dataset_id"],
    ].sort_values(["duplicate_confidence", "duplicate_cluster_id"])
)
display(review.table.loc[review.table["low_pixel_flag"], ["dataset_id", "name", "pixel_count", "duplicate_excluded"]])

available rules: ('high_confidence_duplicates', 'mz_shift_qc_variants', 'explicit_regional_fragments')


,rule,dataset_count
0,high_confidence_duplicates,5
1,mz_shift_qc_variants,0
2,explicit_regional_fragments,0


,duplicate_cluster_id,dataset_id,name,pixel_count,duplicate_confidence,duplicate_excluded,recommended_keeper_dataset_id
57,technical-0005,2024-05-29_16h36m27s,FFGI 20um,360,ambiguous_shared_template,False,<NA>
58,technical-0005,2024-05-29_16h32m16s,FFGI 20um,360,ambiguous_shared_template,False,<NA>
61,technical-0005,2024-05-28_18h53m24s,FFGI 20umEB,360,ambiguous_shared_template,False,<NA>
77,technical-0034,2022-04-16_08h23m55s,2022-04-14_ME_DKFZACLY_S3_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
78,technical-0034,2022-04-16_08h22m38s,2022-04-14_ME_DKFZACLY_S3_W4_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
100,technical-0034,2022-04-16_08h17m46s,2022-04-14_ME_DKFZACLY_S2_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
101,technical-0034,2022-04-16_08h18m19s,2022-04-14_ME_DKFZACLY_S2_W4_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
103,technical-0034,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,10000,ambiguous_shared_template,False,<NA>
22,technical-0069,2026-02-24_17h39m55s,2026_02_24_Rep-01,28745,ambiguous_shared_template,False,<NA>
24,technical-0069,2026-02-23_17h18m48s,2026_02_19_Rep-01,28745,ambiguous_shared_template,False,<NA>


,dataset_id,name,pixel_count,duplicate_excluded
47,2024-06-28_07h10m21s,liver rn lip tic-50ppm,110,True
49,2024-06-26_03h54m35s,liver rn lip tic,110,False
51,2024-06-25_11h23m24s,liver storage day 5 tic,192,False


## 5. Zastosowanie reguł i finalna, poprawiona lista

In [7]:
applied_rules = ["high_confidence_duplicates", "mz_shift_qc_variants", "explicit_regional_fragments"]
explorer.apply_review(review, rules=applied_rules)

final_filters = {
    "organism": "Mouse",
    "organism_part": "Liver",
    "condition": ["Wildtype", "Wtype", "N/A"],
    "polarity": "Negative",
    "mz_min": 200,
    "mz_max": 1400,
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": False,  # see brain_dataset.ipynb section 6 for the cost rationale
}
# exclude_dataset_ids intentionally omitted -- session-level exclusions from steps 2/4 persist across this re-query.
results_liver_repaired = explorer.filter(final_filters)
print(f"repaired liver shortlist: {len(results_liver_repaired)} datasets (previously {len(existing_selected_ids)})")
display(results_liver_repaired[["dataset_id", "name", "analyzer_type", "pixel_count", "molecule_count", "unique_molecule_count"]])

METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

repaired liver shortlist: 18 datasets (previously 18)


,dataset_id,name,analyzer_type,pixel_count,molecule_count,unique_molecule_count
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,Orbitrap,511,207,5
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,Orbitrap,670,37,0
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,Orbitrap,681,109,6
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,Orbitrap,508,182,10
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,Orbitrap,534,225,12
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,Orbitrap,754,262,44
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,Orbitrap,532,120,1
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,Orbitrap,653,216,9
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,Orbitrap,854,829,149
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,Orbitrap,839,468,23


## 6. Które datasety zostały wycięte / dodane względem dotychczasowego `liver/` — pełne porównanie

In [8]:
repaired_selected_ids = set(results_liver_repaired["dataset_id"].astype(str))
review_reasons = {
    dataset_id: "high_confidence_duplicate"
    for dataset_id in review.exclusion_ids(["high_confidence_duplicates"])
}
review_reasons.update({
    dataset_id: "mz_shift_qc_variant"
    for dataset_id in review.exclusion_ids(["mz_shift_qc_variants"])
})

comparison = results[["dataset_id", "name", "condition"]].copy()
comparison["in_original_selection"] = comparison["dataset_id"].isin(existing_selected_ids)
comparison["in_repaired_selection"] = comparison["dataset_id"].isin(repaired_selected_ids)
comparison["status"] = "unchanged_excluded"
comparison.loc[comparison["in_original_selection"] & comparison["in_repaired_selection"], "status"] = "unchanged_included"
comparison.loc[comparison["in_original_selection"] & ~comparison["in_repaired_selection"], "status"] = "REMOVED"
comparison.loc[~comparison["in_original_selection"] & comparison["in_repaired_selection"], "status"] = "ADDED"
comparison["reason"] = comparison["dataset_id"].map(review_reasons).fillna("")

changed = comparison.loc[comparison["status"].isin(["REMOVED", "ADDED"])].sort_values("status")
print(f"datasets whose accept/exclude status changed: {len(changed)}")
display(changed)

print(comparison["status"].value_counts())
print(f"\ntotal: {comparison['in_original_selection'].sum()} (original) -> {comparison['in_repaired_selection'].sum()} (repaired)")

print("\nfor reference -- objectively flagged duplicates outside the final selection")
print("(only relevant if the corpus is later widened beyond the mz_min/mz_max range):")
display(comparison.loc[(comparison["reason"] != "") & ~comparison["in_repaired_selection"]])

datasets whose accept/exclude status changed: 0


,dataset_id,name,condition,in_original_selection,in_repaired_selection,status,reason


status
unchanged_excluded    94
unchanged_included    18
Name: count, dtype: int64

total: 18 (original) -> 18 (repaired)

for reference -- objectively flagged duplicates outside the final selection
(only relevant if the corpus is later widened beyond the mz_min/mz_max range):


,dataset_id,name,condition,in_original_selection,in_repaired_selection,status,reason
46,2024-07-01_09h59m02s,liver storage day 2 tic,Wildtype,False,False,unchanged_excluded,high_confidence_duplicate
47,2024-06-28_07h10m21s,liver rn lip tic-50ppm,Wildtype,False,False,unchanged_excluded,high_confidence_duplicate
48,2024-06-26_03h47m05s,liver storage day 2 tic-110ppm,Wildtype,False,False,unchanged_excluded,high_confidence_duplicate
50,2024-06-25_11h38m02s,liver storage day 2 50ppm,Wildtype,False,False,unchanged_excluded,high_confidence_duplicate
75,2023-06-02_11h54m37s,liver 2-3 50um ms range 300_1500 213_306 pNA-,Wildtype,False,False,unchanged_excluded,high_confidence_duplicate


In [9]:
output_path = Path("data/liver_workspace/configs/datasets/liver_repaired")
exported = explorer.export_selection(output_path, sort_by="download_size_bytes", ascending=False)
exported

{'filters': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/filter.json'),
 'selection': PosixPath('data/liver_workspace/configs/datasets/liver_repaired/selection.json')}

## Podsumowanie

- **Poprawka v2→v3:** filtr biologiczny ujednolicony z brain (`condition=["Wildtype","Wtype","N/A"]`) — pula kandydatów 82 → 112. `mz_min=200, mz_max=1400` bez zmian.
- Ręczny skan gatunek/tkanka-w-nazwie i `(TEST)`: 1 trafienie (`JuS_mouse liver...DAN-test`), sprawdzone i **nie** wykluczone — nazwa jednoznacznie mówi "mouse liver", brak niedopasowania; i tak poza zakresem m/z.
- Przegląd biblioteczny (`profile="liver"`): 5 `high_confidence_duplicates`, 0 `mz_shift_qc_variants`, 0 `explicit_regional_fragments`.
- Wynik: **18 → 18, bez zmian** — żaden z nowo dopuszczonych `N/A`-datasetów ani obiektywnie znalezionych duplikatów nie mieści się w zakresie `200–1400`. Pełna tabela w sekcji 6 to potwierdza wprost (0 `ADDED`, 0 `REMOVED`).
- Eksport do `data/liver_workspace/configs/datasets/liver_repaired/` — istniejący `liver/` nie został nadpisany.